In [ ]:
import pandas as pd
import numpy as np
import glob
from scipy.stats import ttest_ind, ttest_1samp, fisher_exact
import scipy
from tqdm import tqdm


def printShape(df, cols=[], msg=''):
    print(df.shape, end='  ')
    for col in cols:
        print(col, df[col].nunique(), end='  ')
    print(msg, flush=True)
    
    return df

PROJDATA = '/scratch/hi387/SemanticScholar'
CLEANDATA = '../data'
SCISCI = '/scratch/fl1092/SciSciNet/v2'

DATA = '/scratch/fl1092/Email_project/semantic_scholar_data'

In [ ]:
import matplotlib
from matplotlib import pyplot as plt 

cm = 1/2.54  # centimeters in inches
font = {'size': 7}
matplotlib.rc('font', **font)

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

matplotlib.rcParams['grid.linewidth'] = 0.5
matplotlib.rcParams['axes.linewidth'] = 0.5

def set_size(w,h, ax=None):
    """ w, h: width, height in inches """
    if not ax: ax=plt.gca()
    l = ax.figure.subplotpars.left
    r = ax.figure.subplotpars.right
    t = ax.figure.subplotpars.top
    b = ax.figure.subplotpars.bottom
    figw = float(w)/(r-l)
    figh = float(h)/(t-b)
    ax.figure.set_size_inches(figw, figh)


# Classify citation intent

## Stage 1: classification using ChatGPT

In [ ]:
%%time
citations = (
    pd.read_csv(f'{DATA}/CitationContext.csv', usecols=['citingcorpusid','citedcorpusid']).pipe(printShape) # 4105605
    .drop_duplicates().pipe(printShape) # 2011202
)

sampled100k = citations.sample(100000, random_state=0)

sampled100k.to_csv(f'{DATA}/CitationContext_sampled_100k.csv', index=False)

In [ ]:
import csv
import json
import os
from openai import OpenAI

API_KEY = ''

client = OpenAI(api_key=API_KEY)

INPUT_CSV = f'{DATA}/CitationContext_sampled_100k.csv'
OUTPUT_JSONL = f"{DATA}/citation_GPT5mini_labeled_low.jsonl"

TEXT_COLUMN = "contexts"
ID_COLUMN = "ContextID"

MODEL = "gpt-5.4-mini-2026-03-17"

In [ ]:
PROMPT = (
    "Does the sentence itself indicate use of the dataset used in the cited paper?\n"
    "Answer only yes or no.\n"
    "yes = the sentence itself explicitly mentions or clearly describes using the cited paper's dataset.\n"
    "no = otherwise."
)

In [ ]:
def load_done_ids(path):
    done = set()
    if not os.path.exists(path):
        return done
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            obj = json.loads(line)
            done.add(str(obj["row_id"]))
    return done

def main():
    done_ids = load_done_ids(OUTPUT_JSONL)

    with open(INPUT_CSV, "r", encoding="utf-8-sig", newline="") as f_in, \
         open(OUTPUT_JSONL, "a", encoding="utf-8") as f_out:

        reader = csv.DictReader(f_in)

        for i, row in enumerate(reader, start=1):
            
            if i >= 40000: # using the first 40k
                break
                
            row_id = str(row.get(ID_COLUMN) or i)
            if row_id in done_ids:
                continue

            sentence = (row.get(TEXT_COLUMN) or "").strip()
            if not sentence:
                result = {"row_id": row_id, "label": ""}
                f_out.write(json.dumps(result, ensure_ascii=False) + "\n")
                f_out.flush()
                continue

            response = client.responses.create(
                model=MODEL,
                input=[
                    {"role": "system", "content": PROMPT},
                    {"role": "user", "content": sentence},
                ],
                reasoning={"effort": "low"}
            )
    
            in_token = response.usage.input_tokens
            out_token = response.usage.output_tokens
        

            label = response.output_text.strip().lower()
            if label not in {"yes", "no"}:
                # raise ValueError(f"Unexpected label for row {row_id}: {label!r}")
                print(f"Unexpected label for row {row_id}: {label!r}")
            else:

                result = {
                    "row_id": row_id, 'context': sentence,
                    "label": label, 'input_token': in_token,
                    'output_token': out_token
                }
                f_out.write(json.dumps(result, ensure_ascii=False) + "\n")
                f_out.flush()

In [ ]:
main()

## Stage 2: Finetune ModerBERT

### Train

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import torch
import evaluate

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,AutoConfig,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,EarlyStoppingCallback,
    Trainer,
)
from torch.nn import CrossEntropyLoss

In [ ]:
import torch

print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("current device:", torch.cuda.current_device())
    print("device name:", torch.cuda.get_device_name(0))

In [ ]:
# =========================
# Config
# =========================
INPUT_JSONL = f'{DATA}/citation_GPT5mini_labeled_low.jsonl'          # must contain text + label columns
TEXT_COLUMN = "context"           # change if needed
LABEL_COLUMN = "label"             # values: yes / no
MODEL_NAME = "answerdotai/ModernBERT-base" # "bert-base-uncased"
OUTPUT_DIR = f"{DATA}/BERT_citation_intent_data_citation_balanced"
SEED = 42

In [ ]:
# =========================
# Load + prepare data
# =========================
df = pd.read_json(INPUT_JSONL, lines=True).pipe(printShape)#.pipe(generateBalance).pipe(printShape)

df = df[[TEXT_COLUMN, LABEL_COLUMN]].dropna().copy().pipe(printShape)
df[LABEL_COLUMN] = df[LABEL_COLUMN].astype(str).str.strip().str.lower()
df = df[df[LABEL_COLUMN].isin(["yes", "no"])].copy().pipe(printShape)

label_map = {"no": 0, "yes": 1}
df["labels"] = df[LABEL_COLUMN].map(label_map)

train, test = splitData(df, 0.8)
validation, test = splitData(test, 0.5)

train = Dataset.from_pandas(train[[TEXT_COLUMN, "labels"]], preserve_index=False)
validation = Dataset.from_pandas(validation[[TEXT_COLUMN, "labels"]], preserve_index=False)
test = Dataset.from_pandas(test[[TEXT_COLUMN, "labels"]], preserve_index=False)

dataset_dict = DatasetDict({
    "train": train,
    "validation": validation,
    "test": test,
})

print(dataset_dict)

In [ ]:
# =========================
# Compute class weights from training set only
# =========================
train_labels = np.array(dataset_dict["train"]["labels"])
num_no = int((train_labels == 0).sum())
num_yes = int((train_labels == 1).sum())

# inverse-frequency style weights
n_total = len(train_labels)
class_weights = torch.tensor(
    [
        n_total / (2.0 * num_no),   # weight for class 0 = no
        n_total / (2.0 * num_yes),  # weight for class 1 = yes
    ],
    dtype=torch.float,
)

print("Train label counts:", {"no": num_no, "yes": num_yes})
print("Class weights:", class_weights.tolist())

# =========================
# Tokenization
# =========================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# def tokenize_fn(batch):
#     return tokenizer(batch[TEXT_COLUMN], truncation=True)

MAX_LENGTH = 384   # try 128, 256, or 384 first

def tokenize_fn(batch):
    return tokenizer(
        batch[TEXT_COLUMN],
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized = dataset_dict.map(tokenize_fn, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# =========================
# Metrics helpers
# =========================
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")

def metrics_from_preds(preds, labels):
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=preds, references=labels, average="binary")["f1"]
    precision = precision_metric.compute(predictions=preds, references=labels, average="binary")["precision"]
    recall = recall_metric.compute(predictions=preds, references=labels, average="binary")["recall"]
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
    }

# default metric during training: argmax
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metrics_from_preds(preds, labels)

# =========================
# Custom Trainer with weighted loss
# =========================
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = CrossEntropyLoss(
            weight=self.class_weights.to(logits.device)
        )
        loss = loss_fct(
            logits.view(-1, model.config.num_labels),
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss

In [ ]:
# =========================
# Model
# =========================
config = AutoConfig.from_pretrained(MODEL_NAME)
config.reference_compile = False
config.num_labels = 2
config.id2label = {0: "no", 1: "yes"}
config.label2id = {"no": 0, "yes": 1}
config.problem_type = "single_label_classification"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    config=config,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print("using device:", device)
print("model device:", next(model.parameters()).device)

In [ ]:
%%time
# =========================
# Training arguments
# =========================
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=5,
    num_train_epochs=8,
    learning_rate=1e-5,          # lower than 2e-5
    warmup_ratio=0.1,            # smoother start
    weight_decay=0.02,           # a bit stronger than 0.01
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    report_to="none",
    seed=SEED,
    torch_compile=False,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    # callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

# =========================
# Train
# =========================
trainer.train()

In [ ]:
# =========================
# Threshold tuning on dev
# =========================
val_output = trainer.predict(tokenized["validation"])
val_logits = val_output.predictions
val_labels = val_output.label_ids

# binary positive probability = softmax(logits)[:, 1]
val_probs = torch.softmax(torch.tensor(val_logits), dim=-1).numpy()[:, 1]

threshold_rows = []
best_threshold = 0.5
best_f1 = -1.0

for thr in np.arange(0.05, 0.96, 0.05):
    preds = (val_probs >= thr).astype(int)
    m = metrics_from_preds(preds, val_labels)
    row = {"threshold": round(float(thr), 2), **m}
    threshold_rows.append(row)

    if m["f1"] > best_f1:
        best_f1 = m["f1"]
        best_threshold = float(thr)

threshold_df = pd.DataFrame(threshold_rows).sort_values("threshold")
threshold_df.to_csv(os.path.join(OUTPUT_DIR, "dev_threshold_search.csv"), index=False)

print("\nDev threshold search:")
print(threshold_df.to_string(index=False))
print(f"\nBest threshold on dev: {best_threshold:.2f}  |  Best dev F1: {best_f1:.4f}")

In [ ]:
BEST_MODEL_DIR = os.path.join(OUTPUT_DIR, "best_model")

print("Best checkpoint:", trainer.state.best_model_checkpoint)

trainer.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)

print("Saved best model to:", BEST_MODEL_DIR)

### Classify

In [ ]:
import json

MODEL_DIR = f"{DATA}/BERT_citation_intent_data_citation/best_model"

INPUT_CSV = f'{DATA}/CitationContext.csv'
OUTPUT_JSONL = f"{DATA}/predictions_20260405.jsonl"

TEXT_COLUMN = "contexts"
ID_COLUMN = "ContextID"

BATCH_SIZE = 128
MAX_LENGTH = 256
THRESHOLD = 0.5   # only for the emitted yes/no label; keep score for later retuning

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# =========================
# Helpers
# =========================
def load_done_ids(path):
    done = set()
    if not os.path.exists(path):
        return done
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                obj = json.loads(line)
                done.add(str(obj["row_id"]))
    return done

def batched(iterable, batch_size):
    batch = []
    for item in iterable:
        batch.append(item)
        if len(batch) == batch_size:
            yield batch
            batch = []
    if batch:
        yield batch

In [ ]:
# =========================
# Load data
# =========================
df = pd.read_csv(INPUT_CSV)

# Ensure stable row ids
if ID_COLUMN not in df.columns:
    df = df.copy()
    df[ID_COLUMN] = [str(i) for i in range(len(df))]
else:
    df[ID_COLUMN] = df[ID_COLUMN].astype(str)

done_ids = load_done_ids(OUTPUT_JSONL)

# Keep only unprocessed rows
remaining_df = df.loc[~df[ID_COLUMN].isin(done_ids), [ID_COLUMN, TEXT_COLUMN]].copy()

print(f"Total rows: {len(df)}")
print(f"Already done: {len(done_ids)}")
print(f"Remaining: {len(remaining_df)}")

In [ ]:
# =========================
# Inference loop
# =========================
with open(OUTPUT_JSONL, "a", encoding="utf-8") as fout:
    row_iter = remaining_df.itertuples(index=False, name=None)

    for batch in batched(row_iter, BATCH_SIZE):
        row_ids = [str(x[0]) for x in batch]
        texts = [("" if x[1] is None else str(x[1])) for x in batch]

        enc = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )

        enc = {k: v.to(DEVICE) for k, v in enc.items()}

        with torch.no_grad():
            logits = model(**enc).logits
            probs = torch.softmax(logits, dim=-1)[:, 1].detach().cpu().tolist()

        for row_id, text, score in zip(row_ids, texts, probs):
            label = "yes" if score >= THRESHOLD else "no"
            out = {
                "row_id": row_id,
                "label": label,
                "score": float(score),   # positive-class probability
            }
            fout.write(json.dumps(out, ensure_ascii=False) + "\n")

        fout.flush()

print("Done.")

# Process Data Availability Status

In [ ]:
def isUponRequest(text):

    text = text.lower()

    candidates = [
        'available upon request','available on request',
        'available upon reasonable request','available on reasonable request',
        
        'available from the corresponding author upon request','available from the corresponding author on request',
        'available from the corresponding author upon reasonable request',
        'available from the corresponding author on reasonable request',
    ]

    for candi in candidates:
        if candi in text:
            return True
    return False

def isNotApplicable(text):

    text = text.lower()

    candidates = [
        'no data was used', 'no data were used', 'data availability statement: not applicable'
    ]

    for candi in candidates:
        if candi in text:
            return True
    return False

In [ ]:
%%time
dataAva = (
    pd.read_csv(f'{DATA}/DataAvalabilityStatement.csv').pipe(printShape, cols=['corpusid'])
    .dropna().pipe(printShape, cols=['corpusid']) # 227789

    .assign(UponRequest=lambda df: df.data_statement.apply(isUponRequest))
    .assign(NotApplicable=lambda df: df.data_statement.apply(isNotApplicable))

    .query('NotApplicable == False').pipe(printShape, cols=['corpusid']) # 225176
    .drop('NotApplicable', axis=1)
)

In [ ]:
dataAva.UponRequest.value_counts(normalize=True) # 0.264122

# Analysis

## Load data

### Load classified citation intention

In [ ]:
%%time
citationsClassified = (
    pd.read_json(f'{DATA}/predictions_20260405.jsonl', lines=True).pipe(printShape, cols=['row_id'])
    .rename(columns={'row_id':'ContextID'})  # 4105605
)

citations = (
    pd.read_csv(f'{DATA}/CitationContext.csv', usecols=['citingcorpusid','citedcorpusid','ContextID']).pipe(printShape, cols=['ContextID'])

    .merge(citationsClassified, on='ContextID')
    .pipe(printShape, cols=['ContextID'], msg='get citation intention classification') # 4105605
)

del citationsClassified

In [ ]:
# This is the semantic Scholar dataset
citations.head()

In [ ]:
%%time
allCitingCited = (
    pd.concat(
        [
            citations[['citingcorpusid']].rename(columns={'citingcorpusid':'CorpusId'}),
            citations[['citedcorpusid']].rename(columns={'citedcorpusid':'CorpusId'})
        ], ignore_index=True, sort=False
    )
    .drop_duplicates()
    .pipe(printShape, cols=['CorpusId']) # 1606522
)

## Load other processed data

In [ ]:
%%time
allMatchedCountry = pd.read_csv(f'{DATA}/MatchedPaperCountry.csv').pipe(printShape) # 1217308
allMatchedRegionPure = pd.read_csv(f'{DATA}/MatchedPaperSingleSubRegion.csv').pipe(printShape) # 875000

In [ ]:
allMatchedRegion = pd.read_csv(f'{DATA}/MatchedPaperRegion.csv').pipe(printShape, cols=['CorpusId']) # 1342385
# allMatchedRegion = pd.read_csv(f'{DATA}/MatchedPaperSubRegion.csv').pipe(printShape, cols=['CorpusId','PaperID'])

In [ ]:
%%time
paperFieldSubSet = pd.read_csv(f'{DATA}/PaperFieldSubset.csv').pipe(printShape, cols=['corpusid'])
paperYearSubset = pd.read_csv(f'{DATA}/PaperYearSubset.csv').pipe(printShape, cols=['corpusid'])
paperJournalSubset = pd.read_csv(f'{DATA}/PaperJournalSubset.csv').pipe(printShape, cols=['corpusid'])
paperOASubset = pd.read_csv(f'{DATA}/PaperOASubset.csv').pipe(printShape, cols=['corpusid'])

# (1277919, 16)  corpusid 1277919  
# (2208671, 3)  corpusid 1277919  
# (1577204, 2)  corpusid 1577204  
# (1606522, 3)  corpusid 1606522  
# (1605327, 2)  corpusid 1605327  

In [ ]:
paperFieldPivot = paperFieldSubSet.pivot(index='corpusid', columns='category', values='Value').fillna(0)

FOScontrols = ' + '.join([f'C({c})' for c in paperFieldPivot.columns])

paperFieldPivot = paperFieldPivot.reset_index().pipe(printShape, cols=['corpusid'], msg='pivoted') # 1277919

In [ ]:
paperFieldPivot.mean()

## Regression analysis

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

In [ ]:
import math

def lor2p(logit):
    odds = math.exp(logit)
    return odds / (1 + odds)

In [ ]:
citationCounts = (
    citations.groupby(['citedcorpusid']).citingcorpusid.nunique().reset_index()
    .pipe(printShape, cols=['citedcorpusid'])
    .rename(columns={'citedcorpusid':'corpusid', 'citingcorpusid':'CitationCount'})
    
    .merge(dataAva, on='corpusid', how='right')
    .fillna({'CitationCount': 0})
    .drop('data_statement', axis=1)
    .pipe(printShape, cols=['corpusid'])

    .merge(paperYearSubset, on='corpusid', how='left')
    .pipe(printShape, cols=['corpusid'], msg='get paper year')

    .merge(paperFieldPivot, on='corpusid', how='left')
    .pipe(printShape, cols=['corpusid'], msg='get paper field')

    .assign(year=lambda df: df.year.apply(lambda x: x if x>= 2015 and x<= 2025 else np.nan))
    .pipe(printShape, cols=['corpusid'], msg='filter year')
)

dataCitationCounts = (
    citations.query('label == "yes"')
    .groupby(['citedcorpusid']).citingcorpusid.nunique().reset_index()
    .pipe(printShape, cols=['citedcorpusid'])
    .rename(columns={'citedcorpusid':'corpusid', 'citingcorpusid':'CitationCount'})
    
    .merge(dataAva, on='corpusid', how='right')
    .fillna({'CitationCount': 0})
    .assign(IsCited=lambda df: (df.CitationCount > 0).astype(int))
    .drop('data_statement', axis=1)
    .pipe(printShape, cols=['corpusid'])

    .merge(paperOASubset, on='corpusid', how='left')
    .pipe(printShape, cols=['corpusid'], msg='get paper OA')

    .merge(paperJournalSubset, on='corpusid', how='left')
    .pipe(printShape, cols=['corpusid'], msg='get paper journal')

    .merge(paperYearSubset, on='corpusid', how='left')
    .pipe(printShape, cols=['corpusid'], msg='get paper year')

    .merge(paperFieldPivot, on='corpusid', how='left')
    .pipe(printShape, cols=['corpusid'], msg='get paper field')

    .assign(year=lambda df: df.year.apply(lambda x: x if x>= 2015 and x<= 2025 else np.nan))
    .pipe(printShape, cols=['corpusid'], msg='filter year')
)
# (188694, 2)  citedcorpusid 188694  
# (225176, 3)  corpusid 225176  
# (225176, 4)  corpusid 225176  get paper year
# (225176, 19)  corpusid 225176  get paper field
# (225176, 19)  corpusid 225176  filter year
# (22008, 2)  citedcorpusid 22008  
# (225176, 4)  corpusid 225176  
# (225176, 5)  corpusid 225176  get paper OA
# (225176, 7)  corpusid 225176  get paper journal
# (225176, 8)  corpusid 225176  get paper year
# (225176, 23)  corpusid 225176  get paper field
# (225176, 23)  corpusid 225176  filter year